In [ ]:
import pandas as pd

df = pd.read_csv('Coal_Plants_Full_Data_2024.csv')

print(df.head())

   Plant Code         Plant Name              Utility Name  Utility ID State  \
0       703.0              Bowen          Georgia Power Co        7140    GA   
1      6113.0             Gibson  Duke Energy Indiana, LLC       15470    IN   
2      1733.0        Monroe (MI)      DTE Electric Company        5109    MI   
3      3935.0        John E Amos      Appalachian Power Co         733    WV   
4      6002.0  James H Miller Jr          Alabama Power Co         195    AL   

   Total Generators  Total Nameplate Capacity (MW)  \
0                 4                         3498.6   
1                 5                         3339.5   
2                 4                         3279.6   
3                 3                         2932.6   
4                 4                         2822.0   

   Total Summer Capacity (MW)  Total Winter Capacity (MW)  \
0                      3200.0                      3200.0   
1                      3132.0                      3157.0   
2          

In [ ]:
# Install pysr and its Julia backend
!pip install -U pysr
import pysr
pysr.install()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 13.2 MB/s eta 0:00:00
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Using Julia 1.11.5 at /usr/local/bin/julia
[juliapkg] Using Julia project at /root/.julia/environments/pyjuliapkg
[juliapkg] Writing Project.toml:
           | [deps]
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46603ac705cb"
           | Serialization = "9e88b42a-f829-5b0c-bbe9-9e923198166b"
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b

/usr/local/lib/python3.12/dist-packages/pysr/deprecated.py:10: FutureWarning: The `install` function has been removed. PySR now uses the `juliacall` package to install its dependencies automatically at import time. 
  warnings.warn(


In [ ]:
import numpy as np, pandas as pd, math
from pysr import PySRRegressor

CSV = "Coal_Plants_Full_Data_2024.csv"
df = pd.read_csv(CSV)

# ---- Feature engineering matching the DOE / OR-SAGE rubric ----
cap = df["Total Nameplate Capacity (MW)"].fillna(0).astype(float).values
ws  = df["Name of Water Source"].fillna("").str.lower()
dedicated_cooling = ws.str.contains("pond|lake|reservoir|cooling", regex=True).astype(float).values
has_water = (ws.str.len() > 0).astype(float).values
ash       = (df["Ash Impoundment?"].fillna("N") == "Y").astype(float).values
ash_lined = (df["Ash Impoundment Lined?"].fillna("N") == "Y").astype(float).values
ash_penalty = ash * (1.0 - ash_lined)
gv = pd.to_numeric(df["Grid Voltage (kV)"], errors="coerce").fillna(0).values
hv_grid = (gv >= 230).astype(float)

# Population-density proxy: distance to nearest top-50 US metro
metros = [(40.71,-74.01),(34.05,-118.24),(41.88,-87.63),(29.76,-95.37),(33.45,-112.07),
          (39.95,-75.17),(29.42,-98.49),(32.72,-117.16),(32.78,-96.80),(37.34,-121.89),
          (30.27,-97.74),(30.33,-81.66),(37.77,-122.42),(39.96,-82.99),(35.23,-80.84),
          (39.74,-104.99),(35.47,-97.52),(36.16,-86.78),(31.76,-106.49),(36.17,-115.14),
          (38.58,-121.49),(42.36,-71.06),(35.15,-90.05),(38.25,-85.76),(45.52,-122.68),
          (39.29,-76.61),(38.91,-77.04),(45.10,-92.79),(42.33,-83.05),(30.45,-91.19)]
def haversine(la,lo,mla,mlo):
    R=6371.0
    dlat=math.radians(mla-la); dlon=math.radians(mlo-lo)
    a=math.sin(dlat/2)**2+math.cos(math.radians(la))*math.cos(math.radians(mla))*math.sin(dlon/2)**2
    return 2*R*math.asin(math.sqrt(a))
lat = df["Latitude"].astype(float).fillna(0).values
lon = df["Longitude"].astype(float).fillna(0).values
prox = np.array([min(haversine(la,lo,m[0],m[1]) for m in metros) for la,lo in zip(lat,lon)])
pop_ok = np.clip((prox - 6.0) / 34.0, 0.0, 1.0)

cap_norm = np.clip(cap / 1500.0, 0.0, 1.0)
lwr_flag = (cap >= 800).astype(float)

# DOE-rubric ground truth (paper's stated discriminators)
y = np.clip(
    0.45*pop_ok + 0.20*dedicated_cooling + 0.10*has_water +
    0.15*cap_norm + 0.15*lwr_flag + 0.05*hv_grid - 0.10*ash_penalty,
    0, 1
)

X = pd.DataFrame({
    "pop_ok": pop_ok, "cool": dedicated_cooling, "cap": cap_norm,
    "lwr": lwr_flag,  "ash":  ash_penalty,       "hv":  hv_grid,
    "water": has_water,
})

model = PySRRegressor(
    niterations=120,
    binary_operators=["+", "-", "*"],
    unary_operators=[],
    maxsize=20,
    model_selection="best",
    elementwise_loss="L2DistLoss()",
    progress=True,
    random_state=0,
    deterministic=True,
    parallelism='serial',
    procs=0,
)
model.fit(X.values, y, variable_names=list(X.columns))
print("\nBest equation:")
print(model.sympy())

/usr/local/lib/python3.12/dist-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
INFO:pysr.sr:Compiling Julia backend...
/usr/local/lib/python3.12/dist-packages/pysr/sr.py:2890: UserWarning: `numprocs` is specified but will be ignored since `parallelism='serial'`
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 8.000e+04
Progress: 505 / 3720 total iterations (13.575%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.578e-02  0.000e+00  y = 0.71748
3           2.572e-02  1.133e-03  y = water * 0.71913
5           9.191e-03  5.145e-01  y = (cap * 0.34979) + 0.54342
7           7.302e-03  1.151e-01  y = ((lwr + cool) * 0.19363) + 0.63574
9           3.882e-03  3.158e-01  y = ((cool + lwr) * (pop_ok * 0.22603)) + 0.57347
11          3.167e-03  1.018e-01  y = ((water + lwr) + ((cool * 0.64428) + pop_ok)) * 0.2794...
                                      2
13          2.934e-03  3.832e-02  y = (((cool * 0.7059) + (pop_ok * (water + lwr))) * 0.2518...
                                      5) + 0.32209
15          2.929e-03  7.704e-04  y = ((((pop_ok * (water +

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.578e-02  0.000e+00  y = 0.71748
3           2.572e-02  1.133e-03  y = water * 0.71913
5           9.191e-03  5.145e-01  y = (cap * 0.34974) + 0.54344
7           5.086e-03  2.959e-01  y = ((lwr + cool) * 0.22603) + 0.57347
9           3.882e-03  1.350e-01  y = ((pop_ok * (lwr + cool)) * 0.22603) + 0.57347
11          3.167e-03  1.018e-01  y = ((water + lwr) + ((cool * 0.64428) + pop_ok)) * 0.2794...
                                      2
13          2.934e-03  3.832e-02  y = (((cool * 0.7059) + (pop_ok * (water + lwr))) * 0.2518...
                                      5) + 0.32209
15          2.929e-03  7.704e-04  y = ((((pop_ok * (water + lwr)) + (cool * 0.70291)) * 0.37...
                                      776) + 0.48888) * 0.66599
17          2.307e-03  1.193e-01  y = (((lwr + cap) * 0.16548) + 0.63574) - (((pop_ok - 1.44.